# Feature Extraction — Batch Runner

Run `run_extraction` on one or more Kilosort 4 sessions.

**Steps:**
1. Edit the `SESSIONS` list in the configuration cell
2. Run all cells (`Kernel → Restart & Run All`)
3. Check the summary table at the bottom

Output files are written to `output_path` (defaults to `<session_dir>/features/`).

In [ ]:
import sys
from pathlib import Path

# Make the package importable from the notebook
PROJECT_ROOT = Path("../").resolve()   # cerebellum_cell_classifier/
PACKAGE_ROOT = PROJECT_ROOT.parent     # parent that contains the package
if str(PACKAGE_ROOT) not in sys.path:
    sys.path.insert(0, str(PACKAGE_ROOT))

print("Package root:", PACKAGE_ROOT)
print("Project root:", PROJECT_ROOT)

## Configuration

Edit the `SESSIONS` list below. Each entry is a dict with:

| Key | Required | Description |
|-----|----------|-------------|
| `session_path` | **yes** | Kilosort 4 output directory |
| `bin_path` | no | `*ap.bin` file — auto-detected if omitted |
| `output_path` | no | Where to save `.npz` / `.csv` — defaults to `<session>/features/` |
| `labels` | no | Dict of `{unit_id: "CellType"}` expert labels |
| `max_spikes` | no | Max spikes per unit for WF extraction (default 3000) |
| `n_channels_total` | no | Total channels in binary (default 385 for NP2) |
| `sample_rate` | no | Sample rate in Hz (default 30000) |

In [ ]:
# ============================================================
#  EDIT THIS CELL — add / remove sessions as needed
# ============================================================

SESSIONS = [
    {
        "session_path": r"E:\data\AA23\AA23_11",
         "bin_path":   r"E:\data\AA23\AA23_11\original_data\AA23_11_g0_tcat.imec0.ap.bin",  # optional
        "output_path": r"Z:\loco\cell_class\datasets",   # optional
        #"labels": {           # optional — leave as {} if you have no labels yet
            # unit_id: "CellType"
            # 5:   "PC",
            # 124: "MLI",
        #},
    },
    # ── Add more sessions here ──────────────────────────────
    # {
    #     "session_path": r"E:\data\AA24\AA24_01",
    #     "labels": {12: "GC", 88: "PC"},
    # },
]

# ── Global defaults (apply to all sessions unless overridden per-session) ────
DEFAULTS = {
    "n_channels_total": 385,      # NP2: 385  |  NP1: 385  |  3B: 385
    "sample_rate":      30_000.0,
    "max_spikes":       3_000,    # max spikes per unit for WF extraction
}

print(f"{len(SESSIONS)} session(s) configured.")

## Run extraction

In [ ]:
import time
import traceback
import pandas as pd

from cerebellum_cell_classifier.run_extraction import run_extraction

results   = []   # list of returned dicts (one per successful session)
summary   = []   # list of summary rows for the final table

for sess_cfg in SESSIONS:
    # Merge per-session config with global defaults
    cfg = {**DEFAULTS, **sess_cfg}

    session_path = Path(cfg["session_path"])
    labels       = cfg.get("labels") or None     # treat empty dict as None
    bin_path     = cfg.get("bin_path", None)
    output_path  = cfg.get("output_path", None)

    print("\n" + "="*60)
    print(f"  {session_path.name}")
    print("="*60)

    t0 = time.perf_counter()
    try:
        result = run_extraction(
            session_path     = session_path,
            bin_path         = bin_path,
            labels           = labels,
            output_path      = output_path,
            n_channels_total = cfg["n_channels_total"],
            sample_rate      = cfg["sample_rate"],
            max_spikes       = cfg["max_spikes"],
            verbose          = True,
        )
        elapsed = time.perf_counter() - t0
        results.append(result)

        tbl     = result["table"]
        n_units = len(result["unit_ids"])
        n_labelled = int((tbl["label"] != "unknown").sum()) if "label" in tbl.columns else 0

        # Determine where the npz was saved
        out_dir  = output_path or (session_path / "features")
        npz_file = Path(out_dir) / f"{session_path.name}_features.npz"

        summary.append({
            "session":      session_path.name,
            "n_units":      n_units,
            "n_labelled":   n_labelled,
            "elapsed_s":    round(elapsed, 1),
            "status":       "OK",
            "npz":          str(npz_file),
        })

    except Exception as exc:
        elapsed = time.perf_counter() - t0
        print(f"\n[ERROR] {session_path.name}: {exc}")
        traceback.print_exc()
        summary.append({
            "session":    session_path.name,
            "n_units":    0,
            "n_labelled": 0,
            "elapsed_s":  round(elapsed, 1),
            "status":     f"FAILED: {exc}",
            "npz":        "",
        })

print("\n" + "="*60)
print(f"  Finished {len(SESSIONS)} session(s)")
print("="*60)

## Summary

In [ ]:
summary_df = pd.DataFrame(summary)
display(summary_df)

## Quick sanity checks

Per-session label distribution and firing-rate statistics for the successfully processed sessions.

In [ ]:
import numpy as np

for result in results:
    tbl  = result["table"]
    sess = tbl["session"].iloc[0] if "session" in tbl.columns else "?"

    print(f"\n{'='*50}")
    print(f"  {sess}")
    print(f"{'='*50}")
    print(f"  Units : {len(tbl)}")

    if "label" in tbl.columns:
        counts = tbl["label"].value_counts()
        print("  Labels:")
        for lbl, cnt in counts.items():
            print(f"    {lbl:<12} {cnt}")

    if "mean_fr_hz" in tbl.columns:
        fr = tbl["mean_fr_hz"]
        print(f"  FR (Hz): median={fr.median():.1f}  "
              f"min={fr.min():.1f}  max={fr.max():.1f}")

    if "depth_um" in tbl.columns:
        d = tbl["depth_um"]
        print(f"  Depth (um): {d.min():.0f} – {d.max():.0f}")

    # Array shapes
    print(f"  mean_waveforms : {result['mean_waveforms'].shape}")
    print(f"  acg_1d         : {result['acg_1d'].shape}")
    print(f"  acg_3d         : {result['acg_3d'].shape}")

## Launch the viewer

Open the GUI on the first successfully extracted session.  
Pass a different `npz_file` path to open any other session.

In [ ]:
import subprocess, sys

# Pick the npz of the first successful session (change index for a different one)
ok_rows = [r for r in summary if r["status"] == "OK"]
if not ok_rows:
    print("No successful sessions to open.")
else:
    npz_file = ok_rows[0]["npz"]
    viewer   = str(PROJECT_ROOT / "viewer.py")

    print(f"Launching viewer: {npz_file}")
    print("(the viewer runs in a separate process — notebook stays responsive)")

    # Launch as a detached subprocess so the notebook is not blocked
    subprocess.Popen([sys.executable, viewer, npz_file])